# Evaluación de los métodos

## Carga del dataset y calculo de atributos


In [ ]:
import pandas as pd
from load import load_dataset

dataset = load_dataset("futbol_uruguayo.csv")

print(dataset.head())


## División del conjunto en entrenamiento y evaluación

In [ ]:
#separamos cronologicamente el conjunto de entrenamiento y el de evaluacion
#la evaluacion se mantiene separada hasta haber elegido los hiperparametros

train = dataset[
    dataset["date"] < pd.Timestamp("2024-01-01")
].copy()

test = dataset[
    (dataset["date"] >= pd.Timestamp("2024-01-01"))
    & (dataset["date"] < pd.Timestamp("2026-01-01"))
].copy()

print("Cantidad de partidos de entrenamiento:", len(train))
print("Cantidad de partidos de evaluacion:", len(test))

## Creación del pipeline

In [ ]:
import sys
import os

from pipeline import create_model_pipeline, pipeline_input_attributes


# Agrega la carpeta padre (Tarea1) al path de búsqueda de Python
sys.path.append(os.path.abspath(".."))


#hacemos el arbol de decision usando solamente el conjunto de entrenamiento
from decisionTree.classifier import Classifier as DecisionTreeClassifier

treeID3 = DecisionTreeClassifier()

#dejamos fijos los mejores margenes del experimento anterior
model = create_model_pipeline(
    treeID3,
    record_margin=0.00,
    last_matches_margin=0.07,
    goal_difference_margin=0.25,
    attack_margin=0.25,
    defense_margin=0.25,
)

In [ ]:
X_train = train[pipeline_input_attributes].copy()
y_train = train["result"].copy()

X_test = test[pipeline_input_attributes].copy()
y_test = test["result"].copy()

print("Filas de entrenamiento:", len(X_train))
print("Filas reservadas para evaluacion final:", len(X_test))

## División temporal por temporada para Cross-Validation

Se hizo esto por ... TODO: Justificar 

In [ ]:
import numpy as np

#cada temporada se valida usando una ventana de temporadas anteriores
validation_years = [2020, 2021, 2022, 2023]
training_windows = [10, 15]
train_years = train["date"].dt.year.to_numpy()

def create_temporal_splits(window_years):
    temporal_splits = []

    for validation_year in validation_years:
        first_training_year = validation_year - window_years

        fit_indices = np.flatnonzero(
            (train_years >= first_training_year)
            & (train_years < validation_year)
        )
        validation_indices = np.flatnonzero(
            train_years == validation_year
        )

        temporal_splits.append((
            fit_indices,
            validation_indices
        ))

    return temporal_splits


temporal_splits_by_window = {}

for training_window in training_windows:
    temporal_splits_by_window[training_window] = (
        create_temporal_splits(training_window)
    )

    print(f"Ventana de {training_window} temporadas:")

    for validation_year, (fit_indices, validation_indices) in zip(
        validation_years,
        temporal_splits_by_window[training_window]
    ):
        print(
            f"  Validacion {validation_year}:",
            f"entrenamiento={len(fit_indices)},",
            f"validacion={len(validation_indices)}"
        )

In [ ]:
from sklearn.model_selection import GridSearchCV

#buscamos los limites de la tendencia al empate y la ganancia minima
param_grid = {
    "preprocessing__draw_rate__discretizer__low_threshold": [
        0.10,
        0.15,
        0.20,
        0.25,
    ],
    "preprocessing__draw_rate__discretizer__high_threshold": [
        0.30,
        0.35,
        0.40,
        0.45,
    ],
    "model__min_info_gain": [
        0.0025,
        0.0050,
        0.0075,
        0.0100,
    ],
}

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1_macro": "f1_macro",
}

grid_searches = {}
window_results = []

for training_window in training_windows:
    print(f"Buscando con ventana de {training_window} temporadas")

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=scoring,
        refit="f1_macro",
        cv=temporal_splits_by_window[training_window],
        n_jobs=-1,
        verbose=1,
    )

    search.fit(X_train, y_train)
    grid_searches[training_window] = search

    best_index = search.best_index_

    window_results.append({
        "training_window": training_window,
        "accuracy": search.cv_results_["mean_test_accuracy"][best_index],
        "balanced_accuracy": search.cv_results_["mean_test_balanced_accuracy"][best_index],
        "f1_macro": search.cv_results_["mean_test_f1_macro"][best_index],
        "best_params": search.best_params_,
    })

window_results = pd.DataFrame(window_results)
window_results

In [ ]:
#seleccionamos la ventana con mejor F1 macro
best_window_index = window_results["f1_macro"].idxmax()
best_window = int(
    window_results.loc[best_window_index, "training_window"]
)

grid_search = grid_searches[best_window]
temporal_splits = temporal_splits_by_window[best_window]
results = pd.DataFrame(grid_search.cv_results_)

#buscamos automaticamente todas las columnas de hiperparametros
parameter_columns = [
    column
    for column in results.columns
    if column.startswith("param_")
]

score_columns = [
    "split0_test_f1_macro",
    "split1_test_f1_macro",
    "split2_test_f1_macro",
    "split3_test_f1_macro",
    "mean_test_accuracy",
    "mean_test_balanced_accuracy",
    "mean_test_f1_macro",
    "std_test_f1_macro",
    "rank_test_f1_macro",
]

results = results[
    parameter_columns + score_columns
].sort_values("rank_test_f1_macro")

results = results.rename(columns={
    "split0_test_f1_macro": "f1_macro_2020",
    "split1_test_f1_macro": "f1_macro_2021",
    "split2_test_f1_macro": "f1_macro_2022",
    "split3_test_f1_macro": "f1_macro_2023",
})

print(f"Mejor ventana: {best_window} temporadas")
print("Mejores hiperparametros:")
print(grid_search.best_params_)
results.head(10)

In [ ]:
from sklearn.base import clone
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
)
import matplotlib.pyplot as plt
import numpy as np


real_results = []
predicted_results = []

#volvemos a entrenar el mejor modelo para cada division temporal
for fit_indices, validation_indices in temporal_splits:

    fold_model = clone(
        grid_search.best_estimator_
    )

    fold_model.fit(
        X_train.iloc[fit_indices],
        y_train.iloc[fit_indices],
    )

    fold_predictions = fold_model.predict(
        X_train.iloc[validation_indices]
    )

    real_results.extend(
        y_train.iloc[validation_indices]
    )

    predicted_results.extend(
        fold_predictions
    )


real_results = np.asarray(real_results)
predicted_results = np.asarray(predicted_results)

labels = ["L", "E", "V"]

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)

#cantidad de partidos
ConfusionMatrixDisplay.from_predictions(
    real_results,
    predicted_results,
    labels=labels,
    display_labels=[
        "Local",
        "Empate",
        "Visitante",
    ],
    cmap="Blues",
    values_format="d",
    ax=axes[0],
)

axes[0].set_title(
    "Matriz de confusión - cantidades"
)

#porcentaje correcto dentro de cada clase real
ConfusionMatrixDisplay.from_predictions(
    real_results,
    predicted_results,
    labels=labels,
    display_labels=[
        "Local",
        "Empate",
        "Visitante",
    ],
    normalize="true",
    cmap="Blues",
    values_format=".2f",
    ax=axes[1],
)

axes[1].set_title(
    "Matriz de confusión - proporciones"
)

plt.tight_layout()
plt.show()


print(
    classification_report(
        real_results,
        predicted_results,
        labels=labels,
        target_names=[
            "Local",
            "Empate",
            "Visitante",
        ],
        zero_division=0,
    )
)